# 📊 BENCHMARK ASR TIẾNG VIỆT CHO HỆ THỐNG LOCAL HUIT (E1 & RQ1)
---
Theo đúng đặc tả trong **EXPERIMENT_PLAN_VI**:
1. **3 Model ASR bắt buộc:**
   - `vinai/PhoWhisper-base` (Baseline báo cáo)
   - `openai/whisper-small` (Baseline bổ sung 1)
   - `vinai/PhoWhisper-small` (Baseline bổ sung 2)
2. **2 Tập dữ liệu đánh giá:**
   - **VIVOS Test:** Benchmark tiếng Việt công khai chuẩn (460 utterances).
   - **HUIT-Audio:** 120 câu truy vấn miền HUIT (16kHz Mono WAV).
3. **Các chỉ số (Metrics):**
   - **WER** & **CER** kèm **Bootstrap 95% Confidence Interval (CI)**.
   - **RTF (Real-time factor)**, **p50 / p95 Latency (ms)**, **Peak VRAM (MB)**.
4. **Fairness Protocol:** FP16 CUDA, beam_size=5, language='vi', chung text normalizer, 10 samples warm-up.

In [ ]:
# CELL 1: Cài đặt thư viện cần thiết
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa pandas openpyxl tabulate

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU Device: Tesla T4


In [ ]:
# CELL 2: Giải nén tập dữ liệu HUIT-Audio (120 câu) & Chuẩn bị Normalizer
import os, zipfile, glob, re, unicodedata
import numpy as np
import pandas as pd
import evaluate

# 1. Tìm và giải nén file zip HUIT-Audio
zip_files = glob.glob("*.zip")
audio_dir = "huit_audio_16k_wav"
os.makedirs(audio_dir, exist_ok=True)

for z in zip_files:
    print(f"📦 Đang giải nén: {z}...")
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall(".")

found_wavs = glob.glob(f"{audio_dir}/*.wav")
print(f"✅ Đã sẵn sàng {len(found_wavs)} file WAV trong '{audio_dir}'!")

# 2. Text Normalizer dùng chung (Fairness Protocol)
def normalize_vi_text(text: str) -> str:
    if not text or pd.isna(text):
        return ""
    text = unicodedata.normalize("NFC", str(text).strip().lower())
    # Loại bỏ dấu câu thừa, giữ lại chữ cái tiếng Việt và số
    text = re.sub(r"[^\w\s\dàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# 3. Hàm tính Bootstrap 95% Confidence Interval (1000 lần resample)
def calculate_bootstrap_ci(metric_fn, references, hypotheses, n_bootstrap=1000, alpha=0.05):
    scores = []
    n_samples = len(references)
    np.random.seed(42)
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        sub_refs = [references[i] for i in indices]
        sub_hyps = [hypotheses[i] for i in indices]
        scores.append(metric_fn.compute(predictions=sub_hyps, references=sub_refs))
    scores = np.sort(scores)
    lower = np.percentile(scores, 100 * (alpha / 2)) * 100
    upper = np.percentile(scores, 100 * (1 - alpha / 2)) * 100
    return lower, upper

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

📦 Đang giải nén: huit_audio_120.zip...
✅ Đã sẵn sàng 0 file WAV trong 'huit_audio_16k_wav'!


In [ ]:
# CELL 3: Quét tự động toàn bộ file WAV trên Colab và khớp với file Excel
import os, glob
import pandas as pd

# 1. Tìm file Excel metadata
excel_candidates = glob.glob("*HUIT_Audio_120_Queries*.xlsx")
if not excel_candidates:
    raise FileNotFoundError("❌ Chưa tìm thấy file Excel HUIT_Audio_120_Queries.xlsx trên Colab! Hãy upload file này lên nhé.")

excel_file = excel_candidates[0]
print(f"📖 Đang đọc metadata từ: {excel_file}")
df_meta = pd.read_excel(excel_file)

# 2. Quét TẤT CẢ các file .wav đang có trên Colab (dù nằm trong folder nào)
all_wav_files = glob.glob("**/*.wav", recursive=True)
print(f"🔍 Quét được tổng cộng {len(all_wav_files)} file .wav trên toàn hệ thống Colab.")

# Tạo từ điển tên file: { 'huit_001.wav': '/path/to/huit_001.wav' }
wav_map = {os.path.basename(f).strip().lower(): f for f in all_wav_files}

huit_dataset = []
for _, row in df_meta.iterrows():
    raw_name = str(row["Tên File WAV"]).strip().lower()
    if raw_name in wav_map:
        huit_dataset.append({
            "path": wav_map[raw_name],
            "reference": str(row["Nội Dung Cần Đọc (Transcript)"]),
            "speaker": str(row.get("Người Đọc (Speaker)", ""))
        })

print(f"\n🎯 KHỚP THÀNH CÔNG: {len(huit_dataset)}/120 file WAV với nội dung text từ Excel!")

# BẢO VỆ AN TOÀN: Nếu vẫn bằng 0 thì dừng ngay và chỉ rõ nguyên nhân
if len(huit_dataset) == 0:
    print("\n❌ CẢNH BÁO: Chưa tìm thấy file audio nào khớp với danh sách!")
    print("👉 Hãy kiểm tra: Bạn đã upload file zip chứa audio lên chưa và đã chạy Cell 2 giải nén chưa?")
    if len(all_wav_files) > 0:
        print(f"Ví dụ một vài file .wav đang có trên Colab: {all_wav_files[:5]}")
    raise ValueError("Dừng lại vì chưa có audio hợp lệ để benchmark. Hãy giải nén audio trước!")
else:
    print(f"✅ Sẵn sàng benchmark với mẫu đầu tiên: {huit_dataset[0]['path']}")

📖 Đang đọc metadata từ: HUIT_Audio_120_Queries.xlsx
🔍 Quét được tổng cộng 116 file .wav trên toàn hệ thống Colab.

🎯 KHỚP THÀNH CÔNG: 116/120 file WAV với nội dung text từ Excel!
✅ Sẵn sàng benchmark với mẫu đầu tiên: huit_audio_120/huit_001.wav


In [ ]:
# CELL 4: BENCHMARK TRÊN TẬP HUIT-AUDIO (120 CÂU MIỀN HUIT)
import gc, time, soundfile as sf
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

MODELS = [
    {"name": "PhoWhisper-base (Báo cáo)", "id": "vinai/PhoWhisper-base"},
    {"name": "Whisper-small (OpenAI)",     "id": "openai/whisper-small"},
    {"name": "PhoWhisper-small (Bổ sung)", "id": "vinai/PhoWhisper-small"}
]

huit_results = []

for m_cfg in MODELS:
    m_name = m_cfg["name"]
    m_id = m_cfg["id"]
    print(f"\n=================================================================")
    print(f"🚀 Đang benchmark HUIT-Audio với model: {m_name}")
    print(f"=================================================================")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    processor = AutoProcessor.from_pretrained(m_id)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(m_id, torch_dtype=torch_dtype).to(device)
    model.eval()

    gen_kwargs = {"num_beams": 5, "language": "vi", "task": "transcribe"}

    # 1. Warm-up 10 mẫu đầu (không tính vào chỉ số đo)
    print("🔥 Warm-up 10 samples...")
    for item in huit_dataset[:10]:
        audio_arr, sr = sf.read(item["path"])
        feats = processor(audio_arr, sampling_rate=sr, return_tensors="pt").input_features.to(device, dtype=torch_dtype)
        with torch.no_grad():
            _ = model.generate(feats, **gen_kwargs)

    # 2. Chạy đo chính thức
    preds_raw, refs_raw, preds_norm, refs_norm = [], [], [], []
    latencies, audio_durations = [], []

    for idx, item in enumerate(huit_dataset):
        audio_arr, sr = sf.read(item["path"])
        dur = len(audio_arr) / sr
        audio_durations.append(dur)

        feats = processor(audio_arr, sampling_rate=sr, return_tensors="pt").input_features.to(device, dtype=torch_dtype)

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            pred_ids = model.generate(feats, **gen_kwargs)
        torch.cuda.synchronize()
        lat = time.perf_counter() - t0

        p_txt = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

        latencies.append(lat)
        preds_raw.append(p_txt)
        refs_raw.append(item["reference"])
        preds_norm.append(normalize_vi_text(p_txt))
        refs_norm.append(normalize_vi_text(item["reference"]))

        if (idx + 1) % 30 == 0:
            print(f"  Processed {idx + 1}/{len(huit_dataset)} audios...")

    # 3. Tính toán metrics
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    wer = wer_metric.compute(predictions=preds_norm, references=refs_norm) * 100
    cer = cer_metric.compute(predictions=preds_norm, references=refs_norm) * 100
    wer_low, wer_high = calculate_bootstrap_ci(wer_metric, refs_norm, preds_norm)
    cer_low, cer_high = calculate_bootstrap_ci(cer_metric, refs_norm, preds_norm)

    rtf = sum(latencies) / sum(audio_durations)
    p50_lat = np.percentile(latencies, 50) * 1000
    p95_lat = np.percentile(latencies, 95) * 1000

    huit_results.append({
        "Model": m_name,
        "WER (%)": f"{wer:.2f} [{wer_low:.2f}, {wer_high:.2f}]",
        "CER (%)": f"{cer:.2f} [{cer_low:.2f}, {cer_high:.2f}]",
        "RTF": f"{rtf:.3f}",
        "p50 Latency (ms)": f"{p50_lat:.1f}",
        "p95 Latency (ms)": f"{p95_lat:.1f}",
        "Peak VRAM (MB)": f"{peak_vram:.1f}"
    })

    # Lưu file prediction phục vụ Error Analysis (Bảng 8)
    slug = m_name.split()[0].replace('/', '_')
    pd.DataFrame({
        "ref_raw": refs_raw, "pred_raw": preds_raw,
        "ref_norm": refs_norm, "pred_norm": preds_norm,
        "latency": latencies
    }).to_json(f"huit_predictions_{slug}.jsonl", orient="records", lines=True, force_ascii=False)

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()


🚀 Đang benchmark HUIT-Audio với model: PhoWhisper-base (Báo cáo)


Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

🔥 Warm-up 10 samples...


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  Processed 30/116 audios...
  Processed 60/116 audios...
  Processed 90/116 audios...

🚀 Đang benchmark HUIT-Audio với model: Whisper-small (OpenAI)


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

... (23 outputs left out)

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

🔥 Warm-up 10 samples...
  Processed 30/116 audios...
  Processed 60/116 audios...
  Processed 90/116 audios...


In [ ]:
# CELL 5a: NẠP VIVOS TEST (Parquet từ HuggingFace - Cực nhanh, 100% không lỗi)
import io, os
import pandas as pd
import soundfile as sf
from huggingface_hub import hf_hub_download

print("📥 Đang tải VIVOS test dataset từ HuggingFace Parquet...")
# Dùng dataset mirror chính thức dạng Parquet của VIVOS (760 câu test chuẩn)
parquet_file = hf_hub_download(
    repo_id="HaniAI/vivos-vi-asr",
    filename="data/test-00000-of-00001.parquet",
    repo_type="dataset"
)

df_vivos = pd.read_parquet(parquet_file)
print(f"✅ Đã tải xong Parquet! Đang nạp {len(df_vivos)} câu VIVOS test...")

vivos_data = []
text_col = "sentence" if "sentence" in df_vivos.columns else "text"
for _, row in df_vivos.iterrows():
    a = row["audio"]
    if isinstance(a, dict) and a.get("array") is not None:
        arr, sr = a["array"], a["sampling_rate"]
    else:
        arr, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32")
    vivos_data.append({"array": arr, "sr": sr, "sentence": row[text_col]})

print(f"🎯 ĐÃ NẠP THÀNH CÔNG: {len(vivos_data)} câu VIVOS test!")
print(f"Mẫu đầu tiên: '{vivos_data[0]['sentence']}' (sr={vivos_data[0]['sr']})")


[main] parquet test files: []
[refs/convert/parquet] parquet test files: ['default/test/0000.parquet']
📥 Tải default/test/0000.parquet (revision=refs/convert/parquet)


default/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 85.0MB            

default/test/0000.parquet: downloading bytes:           |  0.00B            

✅ Sẵn sàng benchmark 760 câu VIVOS test!
Mẫu: NỬA VÒNG TRÁI ĐẤT HƠN BẢY NĂM | sr = 16000


In [ ]:
# CELL 5b: BENCHMARK VIVOS (lưu kết quả sau mỗi model)
assert len(vivos_data) > 0, "Chạy cell 5a trước để nạp vivos_data"

vivos_results = []

for m_cfg in MODELS:
    m_name, m_id = m_cfg["name"], m_cfg["id"]
    print(f"\n{'='*65}\n🚀 Benchmark VIVOS test với model: {m_name}\n{'='*65}")

    processor = AutoProcessor.from_pretrained(m_id)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        m_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True
    ).to(device)
    model.eval()

    gen_kwargs = {"max_new_tokens": 128, "num_beams": 5, "language": "vi", "task": "transcribe"}

    print("🔥 Warm-up 10 samples...")
    for i in range(min(10, len(vivos_data))):
        feats = processor(vivos_data[i]["array"], sampling_rate=vivos_data[i]["sr"],
                          return_tensors="pt").input_features.to(device, dtype=torch_dtype)
        with torch.no_grad():
            model.generate(feats, **gen_kwargs)

    torch.cuda.reset_peak_memory_stats()
    preds_norm, refs_norm, latencies, audio_durations = [], [], [], []

    for idx, item in enumerate(vivos_data):
        arr, sr = item["array"], item["sr"]
        audio_durations.append(len(arr) / sr)
        feats = processor(arr, sampling_rate=sr, return_tensors="pt").input_features.to(device, dtype=torch_dtype)

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            pred_ids = model.generate(feats, **gen_kwargs)
        torch.cuda.synchronize()
        latencies.append(time.perf_counter() - t0)

        p_txt = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]
        preds_norm.append(normalize_vi_text(p_txt))
        refs_norm.append(normalize_vi_text(item["sentence"]))

        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(vivos_data)} audios...")

    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    wer = wer_metric.compute(predictions=preds_norm, references=refs_norm) * 100
    cer = cer_metric.compute(predictions=preds_norm, references=refs_norm) * 100
    wer_low, wer_high = calculate_bootstrap_ci(wer_metric, refs_norm, preds_norm)
    cer_low, cer_high = calculate_bootstrap_ci(cer_metric, refs_norm, preds_norm)
    rtf = sum(latencies) / sum(audio_durations)

    vivos_results.append({
        "Model": m_name,
        "WER (%)": f"{wer:.2f} [{wer_low:.2f}, {wer_high:.2f}]",
        "CER (%)": f"{cer:.2f} [{cer_low:.2f}, {cer_high:.2f}]",
        "RTF": f"{rtf:.3f}",
        "p50 Latency (ms)": f"{np.percentile(latencies, 50) * 1000:.1f}",
        "p95 Latency (ms)": f"{np.percentile(latencies, 95) * 1000:.1f}",
        "Peak VRAM (MB)": f"{peak_vram:.1f}",
    })
    pd.DataFrame(vivos_results).to_csv("vivos_partial.csv", index=False)  # checkpoint

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

print("\n📊 KẾT QUẢ VIVOS TEST:")
display(pd.DataFrame(vivos_results))


🚀 Benchmark VIVOS test với model: PhoWhisper-base (Báo cáo)


Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔥 Warm-up 10 samples...


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  Processed 100/760 audios...


... (49 outputs left out)

  Processed 700/760 audios...


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


📊 KẾT QUẢ VIVOS TEST:


,Model,WER (%),CER (%),RTF,p50 Latency (ms),p95 Latency (ms),Peak VRAM (MB)
0,PhoWhisper-base (Báo cáo),"8.13 [7.27, 8.95]","3.61 [3.19, 4.03]",0.067,241.4,338.2,258.7
1,Whisper-small (OpenAI),"22.24 [20.91, 23.60]","10.93 [10.09, 11.92]",0.110,402.6,549.6,786.0
2,PhoWhisper-small (Bổ sung),"6.25 [5.56, 6.94]","2.72 [2.38, 3.06]",0.124,447.7,621.5,786.0


In [ ]:
# CELL 6: TỔNG HỢP VÀ XUẤT BẢNG KẾT QUẢ ĐÚNG CHUẨN PAPER
assert "huit_results" in globals() and huit_results, "Chưa có kết quả HUIT, hãy chạy lại cell benchmark HUIT"
assert "vivos_results" in globals() and vivos_results, "Chưa có kết quả VIVOS, hãy chạy xong Cell 5 (nạp dữ liệu + vòng benchmark)"
df_huit_res = pd.DataFrame(huit_results)
df_vivos_res = pd.DataFrame(vivos_results)

print("\n" + "="*80)
print("📋 BẢNG 1: KẾT QUẢ BENCHMARK TRÊN TẬP HUIT-AUDIO (120 CÂU MIỀN HUIT)")
print("="*80)
print(df_huit_res.to_markdown(index=False))
df_huit_res.to_csv("benchmark_huit_audio_summary.csv", index=False)

print("\n" + "="*80)
print("📋 BẢNG 2: KẾT QUẢ BENCHMARK TRÊN TẬP CÔNG KHAI VIVOS TEST")
print("="*80)
print(df_vivos_res.to_markdown(index=False))
df_vivos_res.to_csv("benchmark_vivos_test_summary.csv", index=False)

# Nén toàn bộ file báo cáo kết quả và log để tải về nộp
!zip -q -r asr_benchmark_artifacts.zip *.csv *.jsonl
print("\n📦 ĐÃ XUẤT TẤT CẢ ARTIFACTS: asr_benchmark_artifacts.zip (Sẵn sàng tải về!)")
from google.colab import files
files.download("asr_benchmark_artifacts.zip")


📋 BẢNG 1: KẾT QUẢ BENCHMARK TRÊN TẬP HUIT-AUDIO (120 CÂU MIỀN HUIT)
| Model                      | WER (%)              | CER (%)              |   RTF |   p50 Latency (ms) |   p95 Latency (ms) |   Peak VRAM (MB) |
|:---------------------------|:---------------------|:---------------------|------:|-------------------:|-------------------:|-----------------:|
| PhoWhisper-base (Báo cáo)  | 27.10 [20.18, 35.35] | 19.30 [13.76, 25.81] | 0.079 |              394.3 |              478.7 |            276.9 |
| Whisper-small (OpenAI)     | 33.66 [27.10, 41.21] | 21.61 [16.26, 27.84] | 0.127 |              638.4 |              752.2 |            786   |
| PhoWhisper-small (Bổ sung) | 27.01 [19.64, 35.18] | 19.15 [13.31, 25.63] | 0.147 |              734.5 |              886.7 |            786   |

📋 BẢNG 2: KẾT QUẢ BENCHMARK TRÊN TẬP CÔNG KHAI VIVOS TEST
| Model                      | WER (%)              | CER (%)              |   RTF |   p50 Latency (ms) |   p95 Latency (ms) |   Peak VRAM (MB

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>